# 🧠 Week 07 - Day 05: Training Deep Neural Networks

Welcome to today's session! Today we will learn the theoretical foundations and practical techniques for **training deep neural networks** using **TensorFlow/Keras**.

### 🎯 What You Will Learn Today:
1. 💀 The Vanishing & Exploding Gradients Problem (and how Initialization fixes it).
2. ⚡ Advanced Activation Functions (LeakyReLU).
3. 🛡️ Regularization: Batch Normalization & Dropout.
4. 🚀 Transfer Learning (Using Pre-Trained Models).
5. ⏱️ Callbacks & Learning Rate Scheduling.
6. 🔬 Fine-Tuning a complete Deep Learning Model.


In [1]:
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks, optimizers
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

print(f"TensorFlow Version: {tf.__version__}")

TensorFlow Version: 2.20.0


## 💀 1. The Vanishing & Exploding Gradients Problem
When networks get very deep, gradients can multiply many times during Backpropagation. If the weights are initialized poorly (like all zeros or too small), the gradients become almost zero (**Vanish**). If they are too large, they **Explode**.

> 💡 **Solution: He Normal Initialization** correctly scales starting weights so gradients flow smoothly.

Let's prove this by building two quick models and seeing what happens!


In [3]:
# Let's build a dummy dataset to test initialization
X_dummy = np.random.randn(1000, 20)
y_dummy = np.random.randint(0, 2, 1000)

# Model A: Bad Initialization (Zeros)
model_bad = models.Sequential()
model_bad.add(layers.Dense(64, activation='relu', input_shape=(20,), kernel_initializer='zeros'))
model_bad.add(layers.Dense(64, activation='relu', kernel_initializer='zeros'))
model_bad.add(layers.Dense(1, activation='sigmoid', kernel_initializer='zeros'))

# Model B: Good Initialization (HeNormal)
model_good = models.Sequential()
model_good.add(layers.Dense(64, activation='relu', input_shape=(20,), kernel_initializer='he_normal'))
model_good.add(layers.Dense(64, activation='relu', kernel_initializer='he_normal'))
model_good.add(layers.Dense(1, activation='sigmoid', kernel_initializer='he_normal'))

# Compile both models
model_bad.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model_good.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Notice how Model A (Zero initialization) fails to learn compared to Model B
print("Training Model A (Zero Init):")
hist_bad = model_bad.fit(X_dummy, y_dummy, epochs=5, verbose=0)
print(f"Final Accuracy (Bad): {hist_bad.history['accuracy'][-1]:.2f}")

print("\nTraining Model B (HeNormal Init):")
hist_good = model_good.fit(X_dummy, y_dummy, epochs=5, verbose=0)
print(f"Final Accuracy (Good): {hist_good.history['accuracy'][-1]:.2f}")

c:\Users\asadu\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\layers\core\dense.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Training Model A (Zero Init):
Final Accuracy (Bad): 0.55

Training Model B (HeNormal Init):
Final Accuracy (Good): 0.62


## ⚡ 2. The "Dying ReLU" Problem
Standard `ReLU` turns any negative number strictly into $0$. Sometimes, if a neuron gets a large negative weight, it will **always** output $0$. It becomes "dead" and permanently stops learning.

> 💡 **Solution: Leaky ReLU** allows a tiny, non-zero gradient even for negative numbers!


In [4]:
# Instead of specifying activation='relu' inside the Dense layer, we add it as its own layer!
model_leaky = models.Sequential([
    layers.Dense(32, kernel_initializer='he_normal', input_shape=(20,)),
    layers.LeakyReLU(alpha=0.1), # Alpha is the slope of the negative part
    layers.Dense(16),
    layers.LeakyReLU(alpha=0.1),
    layers.Dense(1, activation='sigmoid')
])
model_leaky.summary()

c:\Users\asadu\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\layers\activations\leaky_relu.py:41: UserWarning: Argument `alpha` is deprecated. Use `negative_slope` instead.
  warnings.warn(


Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_12 (Dense)                │ (None, 32)             │           672 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_re_lu (LeakyReLU)         │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_13 (Dense)                │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_re_lu_1 (LeakyReLU)       │ (None, 16)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_14 (Dense)                │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,217 (4.75 KB)

 Trainable params: 1,217 (4.75 KB)

 Non-trainable params: 0 (0.00 B)

## 🛡️ 3. Regularization: Batch Normalization & Dropout
How do we prevent overfitting and speed up training?
*   **Batch Normalization**: Centers the outputs of a layer so activation functions don't saturate. Speeds up training massively.
*   **Dropout**: Randomly "turns off" a percentage of neurons during each pass. Forces the network to learn robust features since it can't rely on any single neuron!

> ⚠️ **IMPORTANT RULE OF THUMB**: The conventional order to place these layers in Keras is: 
> `Dense` $\rightarrow$ `BatchNormalization` $\rightarrow$ `Activation` $\rightarrow$ `Dropout`


In [5]:
correct_model = models.Sequential([
    # Input Layer
    layers.Dense(128, kernel_initializer='he_normal', input_shape=(100,)),
    
    # 1. Batch Normalization BEFORE Activation
    layers.BatchNormalization(), 
    layers.Activation('relu'),
    
    # 2. Dropout AFTER Activation
    layers.Dropout(0.3), # 30% of neurons turned off
    
    # Hidden Layer 2
    layers.Dense(64, kernel_initializer='he_normal'),
    layers.BatchNormalization(),
    layers.Activation('relu'),
    layers.Dropout(0.3),
    
    # Output classification layer
    layers.Dense(10, activation='softmax')
])

correct_model.summary()

c:\Users\asadu\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\layers\core\dense.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_15 (Dense)                │ (None, 128)            │        12,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation (Activation)         │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_16 (Dense)                │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_1 (Activation)       │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_17 (Dense)                │ (None, 10)             │           650 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 22,602 (88.29 KB)

 Trainable params: 22,218 (86.79 KB)

 Non-trainable params: 384 (1.50 KB)

## 🚀 4. Transfer Learning (Using Pre-Trained Models)
Why build and train heavily from scratch when companies like Google already spent millions doing it for us? 
**Transfer Learning** is the process of taking a massive pre-trained model (like `MobileNetV2`) and adapting its final layers for our own datasets.

Let's download the `CIFAR-10` dataset to see this in action!


In [6]:
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.utils import to_categorical

(x_train, y_train), (x_test, y_test) = cifar10.load_data()

# We normalize pixel values between 0 and 1
x_train = x_train.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0

y_train = to_categorical(y_train, 10)
y_test = to_categorical(y_test, 10)

print("Training Data Shape:", x_train.shape)
print("Testing Data Shape:", x_test.shape)

170498071/170498071 ━━━━━━━━━━━━━━━━━━━━ 495s 3us/step
Training Data Shape: (50000, 32, 32, 3)
Testing Data Shape: (10000, 32, 32, 3)


### Pre-Trained Base Model
Instead of `ResNet50` which takes ages to train in class, we will use `MobileNetV2` today! It was designed to run incredibly fast on mobile phones. Notice how we say `include_top=False` -- we don't want the original 1,000-class head, we will attach our own 10-class head for CIFAR!


In [ ]:
from tensorflow.keras.applications import MobileNetV2

# We set include_top=False to strip the final fully connected layers off
base_model = MobileNetV2(
    input_shape=(32, 32, 3), 
    include_top=False, 
    weights='imagenet' # Download the pre-trained knowledge!
)

# 🥶 FREEZE THE BASE! We don't want to destroy the pre-trained weights yet.
base_model.trainable = False

print(f"Pretrained layers count: {len(base_model.layers)}")

### Building our final Transfer Learning pipeline


In [ ]:
# Data Augmentation integrated right into the sequential model!
data_augmentation = models.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
])

model = models.Sequential([
    # 1. Input Layer explicitly defined
    layers.Input(shape=(32, 32, 3)),
    
    # 2. Augment the image!
    data_augmentation,
    
    # 3. Pass through MobileNet (Frozen)
    base_model,
    
    # 4. Average the 2D maps into 1D arrays
    layers.GlobalAveragePooling2D(),
    
    # 5. Add our Custom Classifier Head!
    layers.BatchNormalization(),
    layers.Dropout(0.5),
    layers.Dense(128, activation='relu', kernel_initializer='he_normal'),
    layers.BatchNormalization(),
    layers.Dropout(0.5),
    layers.Dense(10, activation='softmax')
])
model.summary()

## ⏱️ 5. Callbacks & Learning Rate Scheduling
When training deep neural networks, we want them to stop on their own if they are failing, and we want to dynamically adjust learning rates to squeeze out performance!
*   `EarlyStopping`: Stops training if Validation Loss hasn't improved after `X` epochs.
*   `ModelCheckpoint`: Saves the file only when performance gets better so you never lose the best weights.
*   `ReduceLROnPlateau`: Divides the learning rate by 10 if learning stagnates.


In [ ]:
model.compile(
    optimizer=optimizers.Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Defining our three magical callbacks
# 1. Early Stopping
early_stop = callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

# 2. Model Checkpoint
checkpoint = callbacks.ModelCheckpoint('cifar_best_model.keras', monitor='val_accuracy', save_best_only=True)

# 3. Learning Rate Scheduler
lr_scheduler = callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-6)

print("Ready to train!")

In [ ]:
print("Starting Feature Extraction Training...")
history = model.fit(
    x_train, y_train,
    epochs=10,               # Set max epochs knowing EarlyStopping will save us
    batch_size=64,
    validation_data=(x_test, y_test),
    callbacks=[early_stop, checkpoint, lr_scheduler],
    verbose=1
)

In [ ]:
# Visualize our Training Journey
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Val Accuracy')
plt.title('Accuracy over Time')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.title('Loss over Time')
plt.legend()
plt.show()

## 🔬 6. Fine-Tuning the Deep Network
Right now, the `MobileNetV2` base is strictly frozen. It is just extracting features. But those features were meant for ImageNet, not CIFAR-10 specifically. 

**Fine-tuning** means unfreezing the last few layers of the base model and slowly tweaking them to fit our specific dataset using a microscopically small learning rate!


In [ ]:
# Unfreeze the base model
base_model.trainable = True

# Let's see how many layers are in the base model
print("Number of layers in the base model: ", len(base_model.layers))

# Fine-tune from this layer onwards
fine_tune_at = 100

# Freeze all the layers before the `fine_tune_at` layer
for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False
    
model.summary()

⚠️ **Crucial Detail**: Because we are unfreezing pre-trained weights, we must use an extremely low learning rate (like $1e^{-5}$) so we don't accidentally scramble them and cause Catastrophic Forgetting!


In [ ]:
model.compile(
    optimizer=optimizers.Adam(learning_rate=1e-5), # Notice how tiny the LR is!
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print("Starting Fine-Tuning Training Phase...")
history_fine = model.fit(
    x_train, y_train,
    epochs=5, # Just a few fine-tuning epochs is enough
    batch_size=64,
    validation_data=(x_test, y_test),
    callbacks=[early_stop, checkpoint],
    verbose=1
)

## 🎉 Congratulations!
You have now learned how to tackle Vanishing Gradients, use LeakyReLU, interleave BatchNorm/Dropout, perform feature extraction Transfer Learning, utilize advanced Callbacks, and top it off by Fine-Tuning a Massive Model for extreme performance in TensorFlow!
